In [ ]:
import pandas as pd
import numpy as np
import math
import heapq
import distance_metrics

In [ ]:
#This uses the Iris dataset which has 150 examples and 4 numerical features: sepal length,
#sepal width, petal length, and petal width. Column 0 is a sample ID with no predictive
#value, and the species label is in the last column.
df = pd.read_csv('Iris.csv')
x = df.iloc[:, 1:5]
y = df.iloc[:, 5:]

rows, cols = x.shape

#Shuffling before splitting is important because the Iris CSV is naturally ordered by
#species, which would make a naive slice put all examples of one class into the same split.
shuffle_idx = np.random.permutation(rows)
train_size = int(0.8 * rows)

train_idx = shuffle_idx[:train_size]
test_idx = shuffle_idx[train_size:]

x_train = x.iloc[train_idx].values
x_test = x.iloc[test_idx].values
y_train = y.iloc[train_idx, 0].values
y_test = y.iloc[test_idx, 0].values

print(f"Training samples: {len(x_train)}, Test samples: {len(x_test)}")

In [ ]:
#K is set to an odd number to prevent ties when taking the majority vote across neighbours.
k = 5

#This predicts an example by finding its k nearest neighbours in the training set using
#a given distance function and returning the most common label among them. Rather than
#sorting all distances, a max-heap of size k is used to track the k smallest distances
#seen so far: if a new distance beats the current worst, it gets swapped in.
def predict_one(testing_vector, training_examples, labels, dist_func):

    #Storing all K neighbours in a heap
    heap = []
    for i in range(len(training_examples)):
        dist = dist_func(training_examples[i], testing_vector)
        #Pushing if the heap capacity isn't k
        if len(heap) < k:
            heapq.heappush_max(heap, (dist, i, labels[i]))
        #Popping out the vector with the largest distance so
        #a vector with a smaller distance can be added
        elif dist < heap[0][0]:
            heapq.heappop_max(heap)
            heapq.heappush_max(heap, (dist, i, labels[i]))
    
    #The label with the most votes among the k neighbours is the predicted class.
    votes = {}
    for i in range(len(heap)):
        if heap[i][2] not in votes:
            votes[heap[i][2]] = 1
        else:
            votes[heap[i][2]] += 1

    sorted_map = sorted(votes.items(), key=lambda item:item[1], reverse=True)

    return next(iter(sorted_map))[0]

#Runs predict_one on all test examples and returns the fraction that were correctly classified.
def predict_all(testing_examples, training_examples, training_labels, testing_labels):
    accuracy = 0
    for i in range(len(testing_examples)):
        if(predict_one(testing_examples[i], training_examples, training_labels, dist_func=distance_metrics.Euclidean_dist) == testing_labels[i]):
            accuracy += 1
    
    return accuracy/len(testing_examples)

predict_all(x_test, x_train, y_train, y_test)